### Reproducibility Package for the manuscript *Quantifying the Information Capacity of DNA Methylation as an Epigenetic Memory System*.

Run all cells top to bottom to regenerate every numerical result and table in the manuscript. All output is shown inline; no files are generated by this notebook. Full per-cell detailed description is available in Supplementary Information S4 of the manuscript.

**Cite:** Zenodo Concept DOI `10.5281/zenodo.20633508` (resolves to latest version).

In [1]:
## Environment check and analysis setup - define parameters and functions for 
# information-theoretic calculations and table display.

# Setup and imports
import sys, re, importlib, numpy as np, pandas as pd
pd.set_option('display.float_format', lambda v: f'{v:.2f}')

# ============================================================
#  ENVIRONMENT CHECK
# ============================================================
#  Python
# ============================================================

_py = sys.version_info
assert _py >= (3, 10), f"Python ≥ 3.10 required, got {_py.major}.{_py.minor}"

# ============================================================
# Third-party requirements
# ============================================================

_REQUIRED = {
    "pandas":   (1, 3),   # data manipulation
    "numpy":    (1, 20),  # numerical operations
}

def _parse(ver_str):
    "Numeric version tuple, ignoring pre-release suffixes (e.g. '1.3.0rc1')."
    return tuple(int(x) for x in re.findall(r'\d+', ver_str.split('+')[0])[:3])

_ok, _fail = [], []
for _pkg, _min in _REQUIRED.items():
    try:
        _ver_str = importlib.import_module(_pkg).__version__
        _ver = _parse(_ver_str)
        _min_str = ".".join(map(str, _min))
        if _ver >= _min:
            _ok.append(f"  \u2713  {_pkg:<10} {_ver_str}")
        else:
            _fail.append(f"  \u2717  {_pkg:<10} {_ver_str}  (need \u2265 {_min_str})")
    except ImportError:
        _fail.append(f"  \u2717  {_pkg:<10} NOT INSTALLED  (need \u2265 {'.'.join(map(str, _min))})")

# ============================================================
# Report
# ============================================================

print(f"  \u2713  {'Python':<10} {_py.major}.{_py.minor}.{_py.micro}")
print(*_ok, sep="\n")
print()
if _fail:
    print("  FAILED:")
    print(*_fail, sep="\n")
    raise EnvironmentError("Environment check failed; install missing/outdated packages before running.")
else:
    print("All requirements satisfied. Ready to run.")


N_CPG, P_BIAS = 30e6, 0.9
PHYS_RANGE = (18e6, 27e6)
LC_VALUES  = [1, 5, 10, 20]
K_VALUES   = [100, 1000, 10000, 100000]

# Core information-theoretic functions and 2-decimal table display helper
def shannon_entropy(p):
    p = np.asarray(p, float)
    with np.errstate(divide='ignore', invalid='ignore'):
        h = -p*np.log2(p) - (1-p)*np.log2(1-p)
    return np.where((p == 0) | (p == 1), 0.0, h)

def capacity_mbit(n, h):   return n * float(h) / 1e6    # information capacity, Mbit/cell
def effective_units(n, lc): return n / lc / 1e6         # independent methylation units, millions
def discriminative_bits(k): return float(np.log2(k))    # log2(K), bits; k = number of distinguishable methylation-defined cellular states

def show_table(df, caption):
    floats = df.select_dtypes('float').columns
    return df.style.format('{:.2f}', subset=floats).hide(axis='index').set_caption(caption)

  ✓  Python     3.13.13
  ✓  pandas     3.0.1
  ✓  numpy      2.4.2

All requirements satisfied. Ready to run.


In [2]:
## Reproducibility Module 1 - theoretical and entropy-corrected DNA methylation capacity (Section 2.1 & Table 1)
h05, h09 = float(shannon_entropy(0.5)), float(shannon_entropy(P_BIAS))
lo, hi = capacity_mbit(PHYS_RANGE[0], h09), capacity_mbit(PHYS_RANGE[1], h09)
t1 = pd.DataFrame({
    'Model': ['Theoretical maximum, p = 0.5', 'Biased methylation, p = 0.9',
             'Physiological methylated CpG range, 18-27 million CpGs'],
    'Entropy per CpG site (bits)': [h05, h09, h09],
    'Estimated capacity per cell (Mbit)': [f'~{capacity_mbit(N_CPG, h05):.2f}',
                                 f'~{capacity_mbit(N_CPG, h09):.2f}', f'~{lo:.2f}-{hi:.2f}']})

# Assertions to verify that the calculated values match manuscript values (rounded to 2 decimal places)
assert round(capacity_mbit(N_CPG, h05), 2) == 30.00
assert round(capacity_mbit(N_CPG, h09), 2) == 14.07
assert (round(lo, 2), round(hi, 2)) == (8.44, 12.66)

# Display the final table for theoretical maximum and entropy-corrected DNA methylation information capacity
show_table(t1, 'Table 1. Information capacity of DNA methylation')

Model,Entropy per CpG site (bits),Estimated capacity per cell (Mbit)
"Theoretical maximum, p = 0.5",1.00,~30.00
"Biased methylation, p = 0.9",0.47,~14.07
"Physiological methylated CpG range, 18-27 million CpGs",0.47,~8.44-12.66


In [3]:
## Reproducibility Module 2 - correlation-corrected effective methylation units (Section 2.4 & Table 3)
t3 = pd.DataFrame({
    'Average local correlation length L<sub>c</sub> (CpGs per unit)': LC_VALUES,
    'Effective independent units, N<sub>eff</sub> (millions)': [effective_units(N_CPG, lc) for lc in LC_VALUES],
    'Correlation-corrected information (Mbit)': [capacity_mbit(N_CPG/lc, h09) for lc in LC_VALUES]})

# Assertions to verify that the calculated values match manuscript values (rounded to 2 decimal places)
assert [round(v, 2) for v in t3.iloc[:, 1]] == [30.00, 6.00, 3.00, 1.50]
assert [round(v, 2) for v in t3.iloc[:, 2]] == [14.07, 2.81, 1.41, 0.70]

# Display the final table for correlation-corrected methylation information
show_table(t3, 'Table 3. Correlation-corrected methylation information')

Average local correlation length Lc (CpGs per unit),"Effective independent units, Neff (millions)",Correlation-corrected information (Mbit)
1,30.00,14.07
5,6.00,2.81
10,3.00,1.41
20,1.50,0.70


In [4]:
## Reproducibility Module 3 - discriminative methylation information (Section 2.5 & Table 4; = Table S3.1)
t4 = pd.DataFrame({
    'Biological classification space': ['Broad cell-type atlas', 'Fine cellular subtypes',
        'Cell type x tissue context', 'Cell type x tissue x developmental or functional state'],
    'Number of distinguishable states K': K_VALUES,
    'Upper-bound discriminative information (bits)': [discriminative_bits(k) for k in K_VALUES]})

# Assertion to verify that the calculated values match manuscript values (rounded to 2 decimal places)
assert [round(v, 2) for v in t4.iloc[:, 2]] == [6.64, 9.97, 13.29, 16.61]

# Display the final table for discriminative information
show_table(t4, 'Table 4. Upper-bound discriminative information as a function of cellular-state resolution')

Biological classification space,Number of distinguishable states K,Upper-bound discriminative information (bits)
Broad cell-type atlas,100,6.64
Fine cellular subtypes,1000,9.97
Cell type x tissue context,10000,13.29
Cell type x tissue x developmental or functional state,100000,16.61
